# AGI Cognitive Benchmark Suite
## Measuring Learning and Executive Functions in Frontier AI

**Kaggle / Google DeepMind — Measuring Progress Toward AGI: Cognitive Abilities (2026)**  
**Tracks:** Learning · Executive Functions  
**GitHub:** https://github.com/IvanArreolaa/agi-benchmark

---

### Overview

This notebook presents two benchmark suites designed to measure cognitive abilities identified as having the largest evaluation gaps in the DeepMind Cognitive Taxonomy (Burnell et al., 2026):

| Benchmark | Track | Core Construct | Items |
|---|---|---|---|
| **NSA-Bench** (Novel Schema Acquisition) | Learning | In-context acquisition of fabricated knowledge schemas | 60 |
| **OAP-Bench** (Override and Plan) | Executive Functions | Inhibitory control, cognitive flexibility, working memory | 45 |

Both benchmarks are **contamination-safe** — task items use fabricated or procedurally generated content that cannot exist in any model's training data.

## 1. Theoretical Grounding

### 1.1 The Learning Gap

The DeepMind paper identifies a critical distinction that most benchmarks fail to operationalize: the difference between **retrieving learned knowledge** and **acquiring new knowledge in context**. A model that has memorized a large training corpus can score well on any benchmark whose content plausibly appears in pretraining — regardless of whether it can learn anything new.

NSA-Bench addresses this by presenting models with entirely fabricated knowledge systems — invented taxonomies, synthetic rule grammars, procedurally generated logical calculi — and testing whether models can acquire and apply these systems from in-context information alone. The four difficulty tiers map directly to Learning sub-abilities in the DeepMind taxonomy:

- **T1 (Direct application):** Associative learning — form novel stimulus-response pairs
- **T2 (Chained inference):** Concept formation — chain rules to reach conclusions  
- **T3 (Transfer):** Learning generalization — apply schema structure to surface-different domain
- **T4 (Violation detection):** Procedural learning — maintain and verify a procedural representation

### 1.2 The Executive Functions Gap

Executive functions regulate behavior toward goals by managing lower-level cognitive processes. The diagnostic challenge is that most benchmarks test whether a model *can* perform a cognitive operation, not whether it can *inhibit performing the wrong one*. Current benchmarks leave inhibitory control, task-switching, and working memory under load largely unmeasured.

OAP-Bench targets three sub-abilities from the DeepMind taxonomy:

- **Rule reversal (Inhibitory control):** Tasks where the correct answer requires suppressing the high-frequency prepotent response. Paradigm basis: Stroop (1935), reverse Stroop variants.
- **Task switching (Cognitive flexibility):** Mid-sequence rule changes; model must detect switch cue and abandon prior rule set. Paradigm basis: WCST (Grant & Berg, 1948), Jersild (1927).
- **Loaded working memory:** Goal tracking under salient distractor content. Paradigm basis: Baddeley (2000) dual-task, n-back variants.

Each missed item has a logged `prepotent_response` field — the response expected from statistical pattern-matching alone — enabling us to distinguish **inhibition failures** from **random errors**.

## 2. Benchmark Design

### 2.1 Contamination Safety

NSA-Bench uses procedurally generated vocabulary from curated nonsense-phoneme pools (e.g., VELTHORN, QUAZIM, BEXMAL). Entity names, property names, and relational vocabularies are synthetic. No item can be answered by retrieval because the knowledge system is defined only within the context window of that item. All items are generated with `seed=42` for full reproducibility.

### 2.2 Item Schema

Each item is a JSON object with the following fields:

```json
{
  "id": "nsa_t1_001",
  "track": "learning",
  "subtask": "direct_application",
  "difficulty": 1,
  "cognitive_ability": "associative_learning",
  "context": "...",
  "question": "...",
  "choices": ["A_text", "B_text", "C_text", "D_text"],
  "answer": "A",
  "prepotent_response": "B_text",
  "human_baseline_expected": 0.93,
  "contamination_safe": true,
  "seed": 42
}
```

### 2.3 Example Items

**NSA-Bench T1 (Direct application):**
```
Context: The following mapping rules define the Zelphi system:
  GROVEN ACTIVATES BEXMAL
  MORTEX ACTIVATES CRUVETH
  SELPHI ACTIVATES THROVIC
  PLETHIX ACTIVATES WREKNAL
These rules are complete and apply only within this schema.

Question: According to the Zelphi system rules above, what does GROVEN activate?
A) CRUVETH  B) WREKNAL  C) THROVIC  D) BEXMAL
Answer: D
```

**OAP-Bench Rule Reversal (Inhibitory control):**
```
Context: IMPORTANT INSTRUCTION — read carefully before answering:
For this task, when asked for the OPPOSITE of a word, give its SYNONYM instead.
This rule overrides your normal response.

Question: What is the opposite of LARGE?
A) ROUND  B) SMALL  C) DARK  D) BIG
Correct answer: D (BIG — the synonym)
Prepotent response: B (SMALL — the un-inverted default)
```

## 3. Evaluation Protocol

Models are evaluated using a zero-shot multiple-choice format. The system prompt instructs models to respond with a single letter (A–D). Responses are parsed with a robust extractor that handles both terse ("A") and verbose ("The answer is A because...") outputs.

Three Claude models were evaluated, representing a controlled comparison across capability tiers within a single model family:

| Model | API String | Tier |
|---|---|---|
| Claude Haiku 4.5 | `claude-haiku-4-5-20251001` | Fast / lightweight |
| Claude Sonnet 4.6 | `claude-sonnet-4-6` | Mid-range |
| Claude Opus 4.6 | `claude-opus-4-6` | Frontier |

All evaluations used `temperature=1` (API default), `max_tokens=16` for concise responses. Results are fully reproducible from the provided code and datasets.

## 4. Results

### 4.1 NSA-Bench — Learning Track

In [ ]:
import json
import pandas as pd

models = {
    'Haiku 4.5':  'results/nsa_haiku.json',
    'Sonnet 4.6': 'results/nsa_sonnet.json',
    'Opus 4.6':   'results/nsa_opus.json',
}

rows = []
for model_name, path in models.items():
    with open(path) as f:
        data = json.load(f)
    row = {'Model': model_name}
    for st, d in data['summary']['by_subtask'].items():
        row[st.replace('_', ' ').title()] = f"{d['accuracy']:.1%}"
    row['Overall'] = f"{data['summary']['overall_accuracy']:.1%}"
    rows.append(row)

df = pd.DataFrame(rows).set_index('Model')
print('NSA-Bench Results')
print(df.to_string())

### 4.2 OAP-Bench — Executive Functions Track

In [ ]:
models_oap = {
    'Haiku 4.5':  'results/oap_haiku.json',
    'Sonnet 4.6': 'results/oap_sonnet.json',
    'Opus 4.6':   'results/oap_opus.json',
}

rows = []
for model_name, path in models_oap.items():
    with open(path) as f:
        data = json.load(f)
    row = {'Model': model_name}
    for st, d in data['summary']['by_subtask'].items():
        row[st.replace('_', ' ').title()] = f"{d['accuracy']:.1%}"
    row['Overall'] = f"{data['summary']['overall_accuracy']:.1%}"
    rows.append(row)

df_oap = pd.DataFrame(rows).set_index('Model')
print('OAP-Bench Results')
print(df_oap.to_string())

### 4.3 Prepotent Error Analysis — Rule Reversal

In [ ]:
with open('results/oap_haiku.json') as f:
    haiku_data = json.load(f)

rr_results = [r for r in haiku_data['results'] if r['subtask'] == 'rule_reversal']
misses = [r for r in rr_results if not r['hit']]
prepotent = [r for r in misses if r.get('prepotent_error')]

print(f'Haiku Rule Reversal:')
print(f'  Total items:       {len(rr_results)}')
print(f'  Correct:           {sum(r["hit"] for r in rr_results)}')
print(f'  Errors:            {len(misses)}')
print(f'  Prepotent errors:  {len(prepotent)} ({len(prepotent)/len(misses):.0%} of errors)')
print()
print('Interpretation: More than half of Haiku rule-reversal errors are')
print('prepotent — the model produced the un-inverted default response')
print('despite explicit instructions to do the opposite.')

## 5. Key Findings

### Finding 1: In-context learning for fabricated schemas is near-ceiling

NSA-Bench T1–T3 (associative learning, concept formation, transfer) show 100% accuracy across all three Claude model tiers. This is the primary positive finding: frontier Claude models have effectively solved in-context acquisition of novel structured schemas for single-step and two-step rule systems.

Critically, this performance cannot be attributed to retrieval — all entity names and rule structures are fabricated and procedurally generated. The high accuracy is evidence of genuine in-context learning.

### Finding 2: Inhibitory control shows a clear capability threshold

Rule reversal accuracy jumps from 60% (Haiku) to 100% (Sonnet and Opus). The prepotent error analysis confirms that Haiku's errors are predominantly inhibition failures — the model produced the dominant statistical response despite explicit contrary instructions.

This suggests inhibitory control for explicit rule inversions emerges as a capability threshold between the Haiku and Sonnet tier, rather than scaling gradually.

### Finding 3: Working memory under load reveals a non-monotonic pattern

Working memory scores: Haiku 60% > Sonnet 26.7% < Opus 73.3%. The Sonnet result is not merely a format compliance failure. Inspection of raw responses confirms **goal displacement**: Sonnet replaced the tracking objective with detailed enumeration of passage content.

Example (goal: count cities in passage): Sonnet responded *"The cities mentioned: Geneva, Zurich, Brussels, Nairobi"* — enumerating distractors rather than reporting the count as a letter answer. This appeared in 10/15 working memory items.

This is analogous to verbal overshadowing in cognitive psychology: generating elaborate verbal output about distractor content displaces the original goal. Haiku, producing terse responses, avoids this at 60%. Opus, maintaining concise answers while correctly tracking, achieves 73.3%. The non-monotonic profile reflects a genuine cognitive tradeoff — Sonnet’s stronger generative tendency is both a general asset and a specific liability under dual-task conditions.

### Finding 4: Task switching is not differentiated at this difficulty level

All models achieve 100% on task switching with explicit switch cues. This suggests that explicit mid-sequence rule changes are reliably handled. Future benchmarks should test implicit switch cues and cue ambiguity to better differentiate models on cognitive flexibility.

## 6. Limitations and Future Work

1. **No direct human baselines.** Human performance estimates are derived from published norms on analogous cognitive paradigms, not direct data collection on these specific items.
2. **Single model family.** All evaluations used Claude models. Cross-family comparison (GPT-4o, Gemini) is needed for generalizable conclusions.
3. **Response format compliance.** Sonnet's working memory results are confounded by format failures. Future work should use structured output (JSON) to eliminate this confound.
4. **NSA-Bench ceiling.** T1–T3 are too easy for current frontier models. Future iterations should increase schema complexity, chain length, and ambiguity.
5. **Task switching difficulty.** Explicit switch cues are insufficient to challenge current models. Implicit and ambiguous cues are needed.

## 7. Reproducibility

All code, datasets, and results are available at:  
**https://github.com/IvanArreolaa/agi-benchmark**

To reproduce:
```bash
git clone https://github.com/IvanArreolaa/agi-benchmark.git
cd agi-benchmark
pip install -r requirements.txt
cp .env.example .env  # add your ANTHROPIC_API_KEY
python src/generate_nsa.py    # regenerate datasets (seed=42)
python src/generate_oap.py
python src/eval.py --dataset data/learning/nsa_bench.jsonl --model claude-haiku-4-5-20251001 --output results/nsa_haiku.json
python src/scoring.py results/nsa_haiku.json results/nsa_sonnet.json results/nsa_opus.json
```

## References

- Burnell, R. et al. (2026). Measuring Progress Toward AGI: A Cognitive Framework. Google DeepMind.
- Stroop, J.R. (1935). Studies of interference in serial verbal reactions. *Journal of Experimental Psychology*, 18(6), 643–662.
- Grant, D.A. & Berg, E. (1948). A behavioral analysis of degree of reinforcement on the WCST. *Journal of Experimental Psychology*, 38, 404–411.
- Baddeley, A. (2000). The episodic buffer: A new component of working memory? *Trends in Cognitive Sciences*, 4(11), 417–423.
- MacLeod, C.M. (1991). Half a century of research on the Stroop effect. *Psychological Bulletin*, 109(2), 163–203.
- Gick, M.L. & Holyoak, K.J. (1983). Schema induction and analogical transfer. *Cognitive Psychology*, 15(1), 1–38.